# Adding social and economic data

In [1]:
import pandas as pd


In [2]:
health = pd.read_csv('data/soc-econ/health.csv', sep=';')


selected_indicators = ['Численность лиц в возрасте 18 лет и старше, впервые признанных инвалидами, на 10 000 человек населения соответствующего возраста по субъектам Российской Федерации']
health = health[health["indicator_name"].isin(selected_indicators)]
health = health[health['year'] > 2018]
health = health[["object_name", "year", "indicator_value", "indicator_name"]]
health = health.pivot(index=["object_name", "year"], columns="indicator_name", values="indicator_value").reset_index()
health.rename(columns = {health.columns[2] : "disability_per10k"}, inplace=True)
health = health[["object_name", 'year', 'disability_per10k']]

health

indicator_name,object_name,year,disability_per10k
0,Алтайский край,2019,66.0
1,Алтайский край,2020,54.0
2,Алтайский край,2021,60.3
3,Алтайский край,2022,63.2
4,Амурская область,2019,46.9
...,...,...,...
379,Ямало-Ненецкий автономный округ,2022,28.1
380,Ярославская область,2019,45.8
381,Ярославская область,2020,46.8
382,Ярославская область,2021,41.6


In [3]:
pop = pd.read_csv('data/soc-econ/population.csv', sep=';')

selected_indicators = ['Население субъектов Российской Федерации на 1 января',
                       'Городское население субъектов Российской Федерации на 1 января']

pop = pop[pop["indicator_name"].isin(selected_indicators)]
pop = pop[pop['year'] > 2018]
pop = pop[["object_name", "year", "indicator_value", "indicator_name"]]
pop = pop.pivot(index=["object_name", "year"], columns="indicator_name", values="indicator_value").reset_index()
pop["urban_share"] = pop["Городское население субъектов Российской Федерации на 1 января"]/pop["Население субъектов Российской Федерации на 1 января"]
pop["population"] = pop["Население субъектов Российской Федерации на 1 января"]*1000
pop = pop[["object_name", 'year', 'population', 'urban_share']]
pop

indicator_name,object_name,year,population,urban_share
0,Алтайский край,2019,2250100.0,0.572241
1,Алтайский край,2020,2224100.0,0.575604
2,Алтайский край,2021,2193000.0,0.578796
3,Алтайский край,2022,2154900.0,0.582394
4,Алтайский край,2023,2130900.0,0.583181
...,...,...,...,...
475,Ярославская область,2019,1243800.0,0.814842
476,Ярославская область,2020,1235600.0,0.814098
477,Ярославская область,2021,1221700.0,0.812474
478,Ярославская область,2022,1205600.0,0.810717


In [16]:
emp = pd.read_excel('data/soc-econ/emp2.xlsx')
selected_indicators = ['Уровень безработицы: Уровень безработицы']
emp = emp[emp["indicator_name"].isin(selected_indicators)]
emp = emp[emp["year"] > 2018]
emp = emp[["object_name", "year","indicator_value", "indicator_name"]]
emp = emp.pivot(index=["object_name", "year"], columns="indicator_name", values="indicator_value").reset_index()
emp.rename(columns = {emp.columns[2] : "uneployment"}, inplace=True)
emp

indicator_name,object_name,year,uneployment
0,Алтайский край,2019,5.8
1,Алтайский край,2020,5.9
2,Алтайский край,2021,5.5
3,Алтайский край,2022,3.7
4,Амурская область,2019,5.4
...,...,...,...
379,Ямало-Ненецкий автономный округ,2022,1.7
380,Ярославская область,2019,5.4
381,Ярославская область,2020,7.3
382,Ярославская область,2021,5.9


In [14]:
inc = pd.read_excel('data/soc-econ/inc.xlsx')
selected_indicators = ['Численность населения с денежными доходами ниже границы бедности/величины прожиточного минимума',
                       'Медианный среднедушевой денежный доход населения', 'Потребительские расходы в среднем на душу населения',
                       'Среднедушевые денежные доходы населения']
inc = inc[inc["indicator_name"].isin(selected_indicators)]
inc = inc[inc["year"] > 2018]
inc = inc[["object_name", 'year', "indicator_value", "indicator_name"]]
inc = inc.pivot(index = ['object_name', 'year'], columns= 'indicator_name', values = 'indicator_value').reset_index()
inc.columns = ['object_name', 'year', 'median_income', 'consumption', 'av_income', "share_poverty"]
inc


,object_name,year,median_income,consumption,av_income,share_poverty
0,Алтайский край,2019,18932.4,18389.0,23993.0,17.6
1,Алтайский край,2020,19167.6,17726.0,23917.0,17.5
2,Алтайский край,2021,20783.2,20158.0,26010.0,16.5
3,Алтайский край,2022,24037.0,25062.0,31145.0,15.4
4,Амурская область,2019,25435.7,26407.0,33304.0,15.7
...,...,...,...,...,...,...
379,Ямало-Ненецкий автономный округ,2022,76957.8,47006.0,116639.0,4.5
380,Ярославская область,2019,23190.2,23029.0,28667.0,10.3
381,Ярославская область,2020,24094.4,22727.0,29527.0,9.9
382,Ярославская область,2021,26858.2,27391.0,33124.0,8.9


In [18]:
nat = pd.read_excel("data/soc-econ/ethnic_composition.xlsx", sheet_name=None)
russian_percentages = {}

for region, data in nat.items():
    total_row = data[data.iloc[:, 0].str.contains("Указавшие национальную принадлежность", na=False)]
    total_population = total_row.iloc[0, 1] if not total_row.empty else None
    
    russian_row = data[data.iloc[:, 0].str.startswith("Русские", na=False)]
    russian_population = russian_row.iloc[0, 1] if not russian_row.empty else None
    
    if total_population and russian_population:
        percentage = (russian_population / total_population)*100
        russian_percentages[region] = round(percentage, 2)
    else:
        russian_percentages[region] = "NA"

df_rus = pd.DataFrame.from_dict(russian_percentages, orient='index', columns=['% Russians'])
df_rus.index.name = 'Region'
df_rus.reset_index(inplace=True)

region_name_mapping = {
    'Архангельская область без автономного округа': 'Архангельская область без АО',
    'Ненецкий автономный округ': 'Ненецкий АО',
    'Кемеровская область - Кузбасс': 'Кемеровская область',
    'г. Москва' : 'Москва',
    'г. Санкт-Петербург': 'Санкт-Петербург',
    'г. Севастополь': 'Севастополь',
    'Еврейская автономная область': 'Еврейская АО',
    'Кабардино-Балкарская Республика': 'Кабардино-Балкария',
    'Карачаево-Черкесская Республика': 'Карачаево-Черкесия',
    'Республика Адыгея (Адыгея)': 'Республика Адыгея',
    'Республика Саха (Якутия)': 'Якутия',
    'РСО-Алания': 'Северная Осетия',
    'Республика Татарстан (Татарстан)': 'Республика Татарстан',
    'Чувашская Республика - Чувашия': 'Чувашская Республика',
    'Тюменская область без автономных округов': 'Тюменская область без АО',
    'ХМАО' : 'Ханты-Мансийский АО',
    'ЯНАО' : 'Ямало-Hенецкий АО',
    'Чукотский автономный округ': 'Чукотский АО',
}

df_rus["Region"] = df_rus["Region"].replace(region_name_mapping)

In [21]:
df_se = pop.merge(health, on = ['object_name', 'year'], how = 'left')
df_se = df_se.merge(emp, on = ['object_name', 'year'], how = 'left')
df_se = df_se.merge(inc, on = ['object_name', 'year'], how = 'left')


df_se.rename(columns = {df_se.columns[0] : "Region"}, inplace=True)

region_name_mapping = {
    'Архангельская область без автономного округа': 'Архангельская область без АО',
    'Ненецкий автономный округ': 'Ненецкий АО',
    'Еврейская автономная область':'Еврейская АО',
    'Кемеровская область - Кузбасс': 'Кемеровская область',
    'Город Москва столица Российской Федерации город федерального значения': 'Москва',
    'Город Санкт-Петербург город федерального значения': 'Санкт-Петербург',
    'Город федерального значения Севастополь': 'Севастополь',
    'Еврейская автономная область': 'Еврейская АО',
    'Кабардино-Балкарская Республика': 'Кабардино-Балкария',
    'Карачаево-Черкесская Республика': 'Карачаево-Черкесия',
    'Республика Адыгея (Адыгея)': 'Республика Адыгея',
    'Республика Саха (Якутия)': 'Якутия',
    'Республика Северная Осетия — Алания': 'Северная Осетия',
    'Республика Татарстан (Татарстан)': 'Республика Татарстан',
    'Чувашская Республика - Чувашия': 'Чувашская Республика',
    'Тюменская область без автономных округов': 'Тюменская область без АО',
    'Ханты-Мансийский автономный округ — Югра': 'Ханты-Мансийский АО',
    'Ямало-Ненецкий автономный округ': 'Ямало-Hенецкий АО',
    'Чукотский автономный округ': 'Чукотский АО',
}

df_se["Region"] = df_se["Region"].replace(region_name_mapping)
df_se

,Region,year,population,urban_share,disability_per10k,uneployment,median_income,consumption,av_income,share_poverty
0,Алтайский край,2019,2250100.0,0.572241,66.0,5.8,18932.4,18389.0,23993.0,17.6
1,Алтайский край,2020,2224100.0,0.575604,54.0,5.9,19167.6,17726.0,23917.0,17.5
2,Алтайский край,2021,2193000.0,0.578796,60.3,5.5,20783.2,20158.0,26010.0,16.5
3,Алтайский край,2022,2154900.0,0.582394,63.2,3.7,24037.0,25062.0,31145.0,15.4
4,Алтайский край,2023,2130900.0,0.583181,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
475,Ярославская область,2019,1243800.0,0.814842,45.8,5.4,23190.2,23029.0,28667.0,10.3
476,Ярославская область,2020,1235600.0,0.814098,46.8,7.3,24094.4,22727.0,29527.0,9.9
477,Ярославская область,2021,1221700.0,0.812474,41.6,5.9,26858.2,27391.0,33124.0,8.9
478,Ярославская область,2022,1205600.0,0.810717,40.5,5.0,30524.1,30810.0,38060.0,8.8


In [22]:
df_full = df_rus.merge(df_se, on = 'Region', how = 'left')
df_full.to_csv("intermediate/soc_econ_data.csv")